# EDA A4 – Czy biznes rośnie?

Pytanie biznesowe: **jak zmieniają się zamówienia i przychód w czasie – trend, sezonowość, statusy?**

Z A2 wiemy, że liczba nowych użytkowników jest stała (~1 040 miesięcznie od 2019 r.).
**Jeśli przychód mimo to rośnie, skąd bierze się wzrost?** Sprawdzamy rolę klientów powracających.

Zasady z A1:
* daty bierzemy z **`orders.created_at`** (`order_items.created_at` jest niewiarygodne),
* przychód = Complete + Shipped,
* bieżący, niepełny miesiąc pomijamy w trendach.

## Wczytanie danych

In [ ]:
import pandas as pd
import numpy as np

from bq_utils import read_bq

REVENUE_STATUSES = ["Complete", "Shipped"]

In [ ]:
# Orders

orders = read_bq("orders", usecols=["order_id", "user_id", "status", "created_at", "num_of_item"],
                 dtype={"order_id": "Int32", "user_id": "Int32", "status": "category",
                        "num_of_item": "Int8"})

orders.head()

In [ ]:
# Order items – only order value

order_items = read_bq("order_items", usecols=["order_id", "sale_price"],
                      dtype={"order_id": "Int32"})

order_items.head()

In [ ]:
# Users – signup date, to compare with the first order

users = read_bq("users", usecols=["id", "created_at"], dtype={"id": "Int32"})

users.head()

In [ ]:
# Data as of

checked_at = pd.Timestamp.now(tz="UTC")
current_month = checked_at.tz_localize(None).to_period("M")

print(f"Sprawdzono: {checked_at:%Y-%m-%d %H:%M} UTC, bieżący (pomijany) miesiąc: {current_month}")

## Tabela zamówień do analizy

Jeden wiersz = jedno zamówienie, z dodatkowymi kolumnami:
* `order_value` – suma `sale_price` pozycji,
* `revenue` – wartość zamówienia, jeśli liczy się do przychodu (Complete + Shipped), w przeciwnym razie 0,
* `month`, `year` – z `orders.created_at`,
* `order_rank` – które to zamówienie klienta (1 = pierwsze) → `customer_type`: new / returning.

`order_rank` liczymy na **wszystkich** zamówieniach, zanim odrzucimy bieżący miesiąc.

In [ ]:
# Orders enriched with value, revenue, month and customer type

order_value = order_items.groupby("order_id")["sale_price"].sum().rename("order_value")

orders_v = (orders
            .merge(order_value, left_on="order_id", right_index=True, how="left",
                   validate="one_to_one")
            .fillna({"order_value": 0})
            .assign(is_revenue=lambda x: x["status"].isin(REVENUE_STATUSES),
                    revenue=lambda x: x["order_value"].where(x["is_revenue"], 0),
                    month=lambda x: x["created_at"].dt.tz_localize(None).dt.to_period("M"),
                    order_rank=lambda x: x.sort_values("created_at").groupby("user_id").cumcount() + 1,
                    customer_type=lambda x: np.where(x["order_rank"] == 1, "new", "returning"))
            .assign(year=lambda x: x["month"].dt.year)
            .loc[lambda x: x["month"] < current_month])

print(f"Zamówienia: {len(orders):,} -> do analizy (pełne miesiące): {len(orders_v):,}")
print(f"Przychód: {orders_v['revenue'].sum():,.0f}")

orders_v.head()

## 1. Trend miesięczny

* Liczba zamówień, zamówień liczonych do przychodu, przychód, aktywni klienci, AOV – miesiąc po miesiącu.

In [ ]:
# Monthly KPIs

monthly = (orders_v
           .groupby("month")
           .agg(orders=("order_id", "count"),
                revenue_orders=("is_revenue", "sum"),
                revenue=("revenue", "sum"),
                active_customers=("user_id", "nunique"))
           .assign(aov=lambda x: x["revenue"] / x["revenue_orders"]))

monthly.tail(13).round(1)

## 2. Wzrost rok do roku

Rok 2026 nie jest pełny, więc porównujemy **ten sam okres w każdym roku (YTD)**:
styczeń – ostatni pełny miesiąc. Inaczej 2026 zawsze wyglądałby na spadek.

In [ ]:
# Year-to-date comparison: same months in every year

last_full_month = (current_month - 1).month

ytd = (orders_v[orders_v["month"].dt.month <= last_full_month]
       .groupby("year")
       .agg(orders=("order_id", "count"),
            revenue=("revenue", "sum"),
            active_customers=("user_id", "nunique"))
       .assign(revenue_yoy_pct=lambda x: x["revenue"].pct_change() * 100,
               orders_yoy_pct=lambda x: x["orders"].pct_change() * 100))

print(f"Okres porównania: styczeń–{last_full_month:02d} każdego roku")
ytd.round(1)

## 3. Skąd wzrost? Klienci nowi vs powracający

* Nowi użytkownicy przybywają w stałym tempie, ale każdy z nich może wracać – baza powracających się kumuluje.
* Jaki udział w przychodzie mają zamówienia klientów powracających i jak zmienia się w czasie?

In [ ]:
# Revenue by year: new vs returning customers

(orders_v
 .pivot_table(index="year", columns="customer_type", values="revenue", aggfunc="sum")
 .assign(returning_share_pct=lambda x: x["returning"] / (x["new"] + x["returning"]) * 100)
 .round(1))

In [ ]:
# Time from signup to first order – sanity check (orders before signup would be a data error)

first_order = orders.groupby("user_id")["created_at"].min()
signup = users.set_index("id")["created_at"].reindex(first_order.index)

days_to_first_order = (first_order - signup).dt.total_seconds() / 86400

print(f"Zamówienia przed rejestracją: {(days_to_first_order < 0).mean() * 100:.1f}% kupujących")
days_to_first_order.describe().round(1)

## 4. Sezonowość

* **Indeks sezonowy** = przychód w miesiącu / średni miesięczny przychód w tym samym roku.
  1,10 oznacza miesiąc o 10% lepszy od średniej roku. Uśredniamy po pełnych latach.
* Uwaga: przy silnym trendzie wzrostowym późniejsze miesiące mają indeks zawyżony – to efekt trendu, nie sezonu.
* Dodatkowo: rozkład zamówień wg dnia tygodnia.

In [ ]:
# Seasonal index by calendar month (full years only)

full_years = monthly[monthly.index.year < current_month.year]

seasonal_index = full_years["revenue"] / full_years.groupby(full_years.index.year)["revenue"].transform("mean")

seasonal_index.groupby(seasonal_index.index.month).mean().round(2).rename_axis("month")

In [ ]:
# Orders by day of week (% of all orders)

day_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]

(orders_v["created_at"].dt.day_name()
 .value_counts(normalize=True)
 .reindex(day_order)
 .mul(100)
 .round(1))

## 5. Statusy w czasie

* Czy odsetek anulowań i zwrotów jest stały, czy się zmienia? (temat A5)
* Uwaga: zamówienia z ostatnich miesięcy częściej mają status Processing / Shipped – jeszcze nie dotarły.

In [ ]:
# Status mix by year (% of orders, each row = 100%)

pd.crosstab(orders_v["year"], orders_v["status"], normalize="index").mul(100).round(1)

In [ ]:
# Status mix in the last 6 full months – recent orders are still "in progress"

recent = orders_v[orders_v["month"] >= current_month - 6]

pd.crosstab(recent["month"], recent["status"], normalize="index").mul(100).round(1)

## Wnioski (uzupełnij po uruchomieniu)

1.
2.
3.